# ESN extended sweep: overnight run

Purpose:

1. Run one broader but still controlled ESN sweep on PCA-6 inputs.
2. Test the local parameter trends observed manually.
3. Aggregate across seeds.
4. Save tables for final classical-benchmark assessment.

This is not meant for interactive tuning. Run overnight, inspect summary, then stop classical ESN work unless there is a clear robust improvement.

In [ ]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

print(Path.cwd())

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qpitome_qrc.data.loaders import load_market_stress_data
from qpitome_qrc.baselines.esn import ESNConfig, fit_single_esn
from qpitome_qrc.baselines.esn_benchmark import (
    default_esn_feature_sets,
    build_sequence_splits_for_feature_set,
)
from qpitome_qrc.evaluation.plots import ensure_dir

## 1. Load data and PCA-6 sequence splits

Use PCA-6 and seq_len=20 only. Earlier tests showed seq_len=40 added little and PCA-6 was the best ESN input representation.

In [ ]:
df = load_market_stress_data()
feature_sets = default_esn_feature_sets(df, pca_components=6, corr_threshold=0.95)
pca6_feature_set = [fs for fs in feature_sets if fs.name == "compact_pca_6"][0]

splits20, feature_names = build_sequence_splits_for_feature_set(
    df=df,
    feature_set=pca6_feature_set,
    seq_len=20,
)

print(df.shape)
print(feature_names)
for name, (X, y, dates) in splits20.items():
    print(name, X.shape, dates.min(), dates.max(), "pos_rate", y.mean())

## 2. Define controlled extended sweep

Manual trend hypotheses:

- lower leak rate helped
- higher input scaling helped
- spectral radius increase did not obviously help
- lowering reservoir connectivity did not obviously help

This sweep checks those trends with seed averaging.

In [ ]:
# Adjust these before the overnight run if needed.
SEEDS = (1, 2, 3, 4, 5)
UNITS = (300, 600, 1000)
SPECTRAL_RADIUS = (0.5, 0.7)
LEAK_RATE = (0.03, 0.05, 0.1, 0.2)
INPUT_SCALING = (0.5, 0.6, 0.7, 0.8, 1.0)
INPUT_CONNECTIVITY = (0.3, 0.5, 0.8)
RESERVOIR_CONNECTIVITY = (0.05, 0.1, 0.2)
READOUT_C = (0.03, 0.1, 0.3)
SCALE_STATES = (False, True)

configs = []
for seed in SEEDS:
    for units in UNITS:
        for sr in SPECTRAL_RADIUS:
            for leak in LEAK_RATE:
                for input_scaling in INPUT_SCALING:
                    for input_conn in INPUT_CONNECTIVITY:
                        for reservoir_conn in RESERVOIR_CONNECTIVITY:
                            for readout_c in READOUT_C:
                                for scale_states in SCALE_STATES:
                                    configs.append(
                                        ESNConfig(
                                            units=units,
                                            spectral_radius=sr,
                                            leak_rate=leak,
                                            input_scaling=input_scaling,
                                            input_connectivity=input_conn,
                                            reservoir_connectivity=reservoir_conn,
                                            readout_C=readout_c,
                                            seed=seed,
                                            washout=0,
                                            pooling="final",
                                            scale_states=scale_states,
                                        )
                                    )

print("n_configs:", len(configs))
print("estimated fits =", len(configs))

## 3. Optional smaller dry run

Run this first if you want to verify runtime and output format. Skip for full overnight run.

In [ ]:
DRY_RUN = False

run_configs = configs[:10] if DRY_RUN else configs
print("running configs:", len(run_configs))

## 4. Run sweep

This may take a long time. Intermediate CSV checkpoints are written every 25 runs.

In [ ]:
OUT_DIR = Path("reports/esn_extended_sweep")
TABLE_DIR = OUT_DIR / "tables"
FIGURE_DIR = OUT_DIR / "figures"
ensure_dir(TABLE_DIR)
ensure_dir(FIGURE_DIR)

rows = []
results = {}
start = time.time()

for i, cfg in enumerate(run_configs, start=1):
    print(f"[{i}/{len(run_configs)}] units={cfg.units} sr={cfg.spectral_radius} leak={cfg.leak_rate} scale={cfg.input_scaling} iconn={cfg.input_connectivity} rconn={cfg.reservoir_connectivity} C={cfg.readout_C} seed={cfg.seed} state_scale={cfg.scale_states}")
    res = fit_single_esn(splits=splits20, config=cfg, tune_threshold=True)
    results[i] = res

    rows.append({
        "run_idx": i,
        "units": cfg.units,
        "spectral_radius": cfg.spectral_radius,
        "leak_rate": cfg.leak_rate,
        "input_scaling": cfg.input_scaling,
        "input_connectivity": cfg.input_connectivity,
        "reservoir_connectivity": cfg.reservoir_connectivity,
        "readout_C": cfg.readout_C,
        "seed": cfg.seed,
        "scale_states": cfg.scale_states,
        "threshold": res.threshold,
        "val_pr_auc": res.val_metrics.pr_auc,
        "val_roc_auc": res.val_metrics.roc_auc,
        "val_f1": res.val_metrics.f1_class_1,
        "val_precision": res.val_metrics.precision_class_1,
        "val_recall": res.val_metrics.recall_class_1,
        "test_pr_auc": res.test_metrics.pr_auc,
        "test_roc_auc": res.test_metrics.roc_auc,
        "test_f1": res.test_metrics.f1_class_1,
        "test_precision": res.test_metrics.precision_class_1,
        "test_recall": res.test_metrics.recall_class_1,
    })

    if i % 25 == 0:
        checkpoint = pd.DataFrame(rows)
        checkpoint.to_csv(TABLE_DIR / "esn_extended_sweep_checkpoint.csv", index=False)

elapsed = time.time() - start
summary = pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)
summary.to_csv(TABLE_DIR / "esn_extended_sweep_runs.csv", index=False)

print("elapsed minutes:", elapsed / 60)
summary.head(20)

## 5. Aggregate across seeds

This is the model-selection table. Do not select by single seed unless explicitly documenting exploratory behavior.

In [ ]:
group_cols = [
    "units",
    "spectral_radius",
    "leak_rate",
    "input_scaling",
    "input_connectivity",
    "reservoir_connectivity",
    "readout_C",
    "scale_states",
]

agg = (
    summary
    .groupby(group_cols, as_index=False)
    .agg(
        mean_val_pr_auc=("val_pr_auc", "mean"),
        std_val_pr_auc=("val_pr_auc", "std"),
        mean_val_f1=("val_f1", "mean"),
        mean_val_precision=("val_precision", "mean"),
        mean_val_recall=("val_recall", "mean"),
        mean_test_pr_auc=("test_pr_auc", "mean"),
        mean_test_f1=("test_f1", "mean"),
        mean_test_precision=("test_precision", "mean"),
        mean_test_recall=("test_recall", "mean"),
        n_seeds=("val_pr_auc", "size"),
    )
    .sort_values("mean_val_pr_auc", ascending=False)
    .reset_index(drop=True)
)

agg.to_csv(TABLE_DIR / "esn_extended_sweep_seed_aggregate.csv", index=False)
agg.head(30)

## 6. Robustness filters

A good candidate should improve validation without completely collapsing precision or test behavior.

In [ ]:
filtered = agg[
    (agg["n_seeds"] >= len(SEEDS))
    & (agg["mean_val_pr_auc"] >= 0.58)
    & (agg["mean_val_precision"] >= 0.45)
    & (agg["mean_test_precision"] >= 0.30)
]

filtered.head(30)

## 7. Visual summaries

In [ ]:
plot_df = agg.sort_values("mean_val_pr_auc", ascending=True).tail(30).copy()
labels = (
    "u=" + plot_df["units"].astype(str)
    + " leak=" + plot_df["leak_rate"].astype(str)
    + " scale=" + plot_df["input_scaling"].astype(str)
    + " iconn=" + plot_df["input_connectivity"].astype(str)
    + " rconn=" + plot_df["reservoir_connectivity"].astype(str)
    + " C=" + plot_df["readout_C"].astype(str)
    + " ss=" + plot_df["scale_states"].astype(str)
)

fig, ax = plt.subplots(figsize=(15, max(7, 0.45 * len(plot_df))))
ax.barh(labels, plot_df["mean_val_pr_auc"], xerr=plot_df["std_val_pr_auc"].fillna(0))
ax.axvline(0.578, linestyle="--", label="toy tabular reference ~0.578")
ax.axvline(0.600, linestyle=":", label="target 0.60")
ax.set_xlabel("Mean validation PR-AUC across seeds")
ax.set_title("Extended ESN sweep: top aggregated configs")
ax.grid(axis="x", alpha=0.25)
ax.legend(loc="lower right")
ax.tick_params(axis="y", labelsize=8)
plt.subplots_adjust(left=0.52, right=0.97, top=0.93, bottom=0.08)
plt.savefig(FIGURE_DIR / "top_aggregated_configs.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(agg["mean_val_pr_auc"], agg["mean_test_pr_auc"], alpha=0.6)
ax.axvline(0.578, linestyle="--", label="toy tabular val ref")
ax.axvline(0.600, linestyle=":", label="target val")
ax.set_xlabel("Mean validation PR-AUC")
ax.set_ylabel("Mean test PR-AUC")
ax.set_title("Validation vs test PR-AUC across ESN configs")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "val_vs_test_pr_auc.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Final decision notes

Suggested stopping rules:

- If best seed-averaged validation PR-AUC is below 0.58, keep toy tabular baseline as stronger classical baseline and stop ESN tuning.
- If best seed-averaged validation PR-AUC is 0.58–0.62, document ESN as a weak but honest temporal benchmark.
- If best seed-averaged validation PR-AUC exceeds 0.62 and test behavior does not collapse, use it as the main classical temporal benchmark for QRC comparison.
- If validation/test scatter shows no correlation, regime mismatch dominates and walk-forward validation is needed later.

For the current project phase, this should close classical ESN work and shift focus to QRC unless the overnight sweep produces a robust surprise.